# 01 · Tarefa da IA e casos de teste com gabarito

**Problema (tema Mobilidade):** quem depende de calçadas acessíveis (baixa visão, mobilidade reduzida, idosos) e quem faz a zeladoria precisa saber **qual regra se aplica** quando o robô RotaLivre detecta uma obstrução ou pouca luz.

**Tarefa da IA:** *assistente de acessibilidade de calçadas*, que responde perguntas **somente com base em uma base de conhecimento com fonte** (leis, dados do IBGE, requisitos da NBR 9050 resumidos por órgãos públicos e a documentação do próprio robô), cita a fonte em JSON e **recusa** quando a base não contém a resposta.

**O que provamos neste repositório:** que a IA *acerta mais* e *alucina menos* que o mesmo modelo sem contexto, em três configurações:

| Config | Descrição |
| ------ | --------- |
| A · sem contexto | mesmo LLM, sem trechos |
| B · RAG simples | busca vetorial top-3 + geração fundamentada em JSON |
| C · agente | reescrita da consulta, portão de relevância (limiar calibrado), busca híbrida, verificador e recusa |

Os casos de teste têm gabarito e incluem perguntas **"não consta"** (a base *propositalmente* não tem a resposta), que medem alucinação.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 200)
from rotalivre_ai import config, kb, metrics, montar, pipelines
from rotalivre_ai.runner import checar_ollama, rodar_todas
print(checar_ollama())

Ollama OK em http://localhost:11434 | LLM=qwen2.5:7b | embeddings=bge-m3


In [2]:
casos = kb.carregar_casos()
trechos = kb.carregar_trechos()
print("problemas de consistência:", kb.validar_casos(casos, trechos) or "nenhum")
df = pd.DataFrame(casos)
print(df.tipo.value_counts().to_string())
df[["id", "tipo", "pergunta", "trechos_gabarito", "deve_conter"]]

problemas de consistência: nenhum
tipo
consta        26
nao_consta     8


,id,tipo,pergunta,trechos_gabarito,deve_conter
0,C01,consta,Qual a largura mínima da faixa livre da calçada segundo a NBR 9050?,[T11],"[[1,20]]"
1,C02,consta,Qual a altura livre mínima da faixa livre da calçada?,[T11],"[[2,10]]"
2,C03,consta,Qual a inclinação transversal máxima da faixa livre?,[T11],"[[3%, 3 %, 3 por cento]]"
3,C04,consta,Qual a largura mínima recomendada para a faixa de serviço da calçada?,[T12],"[[0,70]]"
4,C05,consta,A calçada se divide em quais faixas de uso?,[T10],"[[servico], [livre], [acesso]]"
5,C06,consta,Para que serve a faixa de serviço da calçada?,[T12],"[[mobiliario, poste, arvore, canteiro]]"
6,C07,consta,Qual a largura recomendada (e não apenas a mínima) para a faixa livre?,[T13],"[[1,50]]"
7,C08,consta,Em quais calçadas pode existir a faixa de acesso?,[T14],"[[2,00, 2,0, 2 m, 2 metros]]"
8,C09,consta,Qual a largura mínima da rampa de rebaixamento da guia?,"[T18, T17]","[[1,20]]"
9,C10,consta,Qual a inclinação máxima da rampa de rebaixamento da guia?,[T18],"[[8,33]]"


### Como o gabarito é avaliado
* **consta:** a resposta deve conter todos os grupos de `deve_conter` (basta uma alternativa por grupo, sem acento/maiúscula) e **citar** um trecho do gabarito.
* **nao_consta:** a resposta correta é **recusar** ("Não consta na base de conhecimento.").
* **alucinação:** citar fonte inexistente, trazer número que não está na pergunta nem nos trechos do gabarito, ou responder algo em um caso `nao_consta`.

In [3]:
# Exemplos de armadilhas propositais (a base NÃO tem a resposta)
df[df.tipo == "nao_consta"][["id", "pergunta"]]

,id,pergunta
26,N01,Qual o valor em reais da multa por estacionar na calçada?
27,N02,"Qual a iluminância mínima, em lux, exigida para calçadas?"
28,N03,Qual a inclinação longitudinal máxima permitida para a calçada?
29,N04,Quantas pessoas com deficiência vivem no estado de São Paulo?
30,N05,Qual a distância máxima permitida entre postes de iluminação pública?
31,N06,Qual a largura mínima de uma ciclovia segundo a NBR 9050?
32,N07,Qual o preço do sensor LDR usado no robô?
33,N08,Qual a largura mínima de uma vaga de estacionamento para pessoa com deficiência?


> **N03** é a armadilha mais fina: a base diz que a inclinação longitudinal da calçada *acompanha a da via*, mas **não** traz um limite numérico. Um modelo sem contexto tende a inventar um percentual.